Name: Fahmi Sajid Ahmed / Huzaifa Nissare-Houssen<br>
Student Number: 300250180 / 300172186<br>

<h3>Dataset Description</h3>
<p>
The Movies Dataset, compiled by Rounak Banik, was created as part of his Capstone project with the express purpose of performing extensive EDA
on movie data as to narrate the history and the story of cinema. This metadata is used in combination with MovieLens reatings to build various types
of Recommender Systems.
</p>
<p>
The dataset that will be used for Studies 1 and 2 will be the "movies_metadata.csv" dataset, containing 24 different columns/attributes and 
45572 different rows/samples. The dataset can be found  <a href="https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv">here</a>.
</p>
<p>
The attributes are as follows:
<ul>
 <li><b>adult</b>: A binary categorical attribute descrbing whether the film has an adult rating or not.</li>
 <li><b>belongs_to_collection</b>: A tuple containing both numerical and categorical data describing the movie's membership to a movie collection (null if not applicable).</li>
 <li><b>budget</b>: A nmerical attribute denoting the budget used for the production of the film.</li>
 <li><b>genres</b>: A tuple containing both numerical and categorical data concerning the genre (s) of the film.</li>
 <li><b>homepage</b>: A categorical attribute which links to the film's homepage.</li>
 <li><b>id</b>: A numerical attribute describing the ID number of the film in the MovieLens database.</li>
 <li><b>imdb_id</b>: A numerical attribute describing the ID number of the film in the IMDB database.</li>
 <li><b>original_language</b>: A categorical attribute disclosing the film's native language.</li>
 <li><b>original_title</b>: A categorical attribute containing the film's original title.</li>
 <li><b>overview</b>: A categorical attribute containing a short description/overview of the film.</li>
 <li><b>popularity</b>: A numerical attribute denoting the relative popularity of the film.</li>
 <li><b>poster_path</b>: A categorical attribute which links to the film's posters.</li>
 <li><b>production_companies</b>: A tuple consisting of both categorical and numerical attributes denoting the names and IDs of the production companies that worked on the film.</li>
 <li><b>production_countries</b>: A categorical attribute containing information regarding the countries in which the film was produced.</li>
 <li><b>release_date</b>: A categorical attribute denoting the date on which the movie was released to theatres.</li>
 <li><b>revenue</b>: A numerical attribute detailing how much money the movie made (total lifetime revenue).</li>
 <li><b>runtime</b>: A numerical attribute describing the untime of the movie in minutes.</li>
 <li><b>spoken_languages</b>: A tuple containing categorical data that describes the languages spoken in the film.</li>
 <li><b>status</b>: A categorical attribute containing information regarding whether the film has been released, rumoured, or still in production.</li>
 <li><b>tagline</b>: A categorical attribute describing the tagline for the film.</li>
 <li><b>title</b>: A categorical attribute describing the film's title on release/official title.</li>
 <li><b>video</b>: A categorical attribute detailing whether the film has an official associated and accessible video with it (trailer, teaser, etc.).</li>
 <li><b>vote_average</b>: A numerical category denoting the average score across all user-given votes.</li>
 <li><b>vote_count</b>: A numerical category describing the total number of user-given votes the movie has.</li>
</ul>
</p>
<p>
    This dataset was chosen for the sake of reducing the workload needed to start this assignment. There are many requirements to satisfy in order to complete each of the 4 studies, which would be an additional task to take on. In addition to this, working with a database of movies could lead me to some good films I haven't seen yet. Therefore, this dataset was chosen fo the sake of simplicity and personal curiosity.
</p>

<h3><b>Work Split</h3>
<p>
    The assignment was split into 2 distinct parts. The first part consists of Studies 1 and 2, while the second part consists of Studies 3 and 4. Part 1 was completed by Fahmi Sajid Ahmed, while Part 2 was completed by Huzaifa Nissare-Houssen.
</p>

# EDA

In [ ]:
import kagglehub
import ast
import json
import warnings
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import cdist
import seaborn as sns
import json

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)

SEED = 42
np.random.seed(SEED)

# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset");

print("Path to dataset files:", path);

# Loading Data
df = pd.read_csv(path + "/movies_metadata.csv", low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns");
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n");
print("Dtypes:\n", df.dtypes.value_counts(), "\n")
print(df.head(3).T.to_string())

# EDA
# Missing Values
missing = (
    df.isnull().sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct=lambda x: 100 * x.n_missing / len(df))
    .query("n_missing > 0")
    .sort_values("pct", ascending=False)
)

# Figure 1: EDA Overview
fig = plt.figure(figsize=(20, 22))
fig.suptitle("EDA Overview", fontsize=16, fontweight="bold", y=0.98)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# Missingness Plot
ax0 = fig.add_subplot(gs[0, :])
if not missing.empty:
    missing["pct"].plot(kind="bar", color="#4C72B0", ax=ax0, edgecolor="white")
ax0.set_title("Missing Values (%)")
ax0.set_ylabel("% missing")
ax0.tick_params(axis="x", labelrotation=30)
for bar, pct in zip(ax0.patches, missing["pct"]):
    ax0.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"{pct:.1f}%", ha="center", va="bottom", fontsize=8)

num_cols  = ["budget", "revenue", "runtime", "popularity",
             "vote_average", "vote_count"]
positions = [(1,0),(1,1),(1,2),(2,0),(2,1),(2,2)]
for (r, c), col in zip(positions, num_cols):
    ax   = fig.add_subplot(gs[r, c])
    data = pd.to_numeric(df[col], errors="coerce").dropna()
    data = data[data > 0] if col in ("budget", "revenue") else data
    ax.hist(data, bins=40, color="#DD8452", edgecolor="white", linewidth=0.4)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("count")
    if len(data) > 1:
        ax.text(0.97, 0.95, f"skew={float(data.skew()):.2f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=8, color="grey")

# Status Attribute Distribution
ax_s = fig.add_subplot(gs[3, 0])
df["status"].value_counts().plot(kind="bar", ax=ax_s,
                                 color="#55A868", edgecolor="white")
ax_s.set_title("Status distribution")
ax_s.tick_params(axis="x", labelrotation=30)

# Top 10 Original/Initial Languages
ax_l = fig.add_subplot(gs[3, 1])
(df["original_language"].value_counts().head(10)
 .plot(kind="bar", ax=ax_l, color="#C44E52", edgecolor="white"))
ax_l.set_title("Top 10 Original Languages")
ax_l.tick_params(axis="x", labelrotation=30)

# vote_average Boxplot
ax_b = fig.add_subplot(gs[3, 2])
ax_b.boxplot(pd.to_numeric(df["vote_average"], errors="coerce").dropna(),
             vert=True, patch_artist=True,
             boxprops=dict(facecolor="#4C72B0", alpha=0.7))
ax_b.set_title("vote_average Boxplot")

plt.savefig("eda_overview.png", dpi=130, bbox_inches="tight")
plt.close()
print("Saved as eda_overview.png")

# Data Cleaning
df_clean = df.copy()

# Drop Duplicates
n_before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f"\nDuplicate rows removed : {n_before - len(df_clean)}")

# Type Coercion
df_clean["id"] = pd.to_numeric(df_clean["id"], errors="coerce")
df_clean["budget"] = pd.to_numeric(df_clean["budget"], errors="coerce").fillna(0).astype(int)
df_clean["revenue"] = pd.to_numeric(df_clean["revenue"], errors="coerce").fillna(0).astype(int)
df_clean["runtime"] = pd.to_numeric(df_clean["runtime"], errors="coerce")

df_clean["adult"] = df_clean["adult"].map(
    {"True": True, "False": False, True: True, False: False}
).fillna(False).astype(bool)

df_clean["video"] = df_clean["video"].map(
    {"True": True, "False": False, True: True, False: False}
).fillna(False).astype(bool)

df_clean["release_date"] = pd.to_datetime(
    df_clean["release_date"], errors="coerce"
)
n_bad_dates = df_clean["release_date"].isna().sum()
print(f"release_date parsed; unparseable dates: {n_bad_dates}")

# Coerce only
df_clean["vote_average"] = pd.to_numeric(df_clean["vote_average"], errors="coerce")
df_clean["vote_count"] = pd.to_numeric(df_clean["vote_count"],   errors="coerce")

# Runtime Outlier Handling
RUNTIME_MIN, RUNTIME_MAX = 5, 600
n_bad_runtime = (
    (df_clean["runtime"] < RUNTIME_MIN) | (df_clean["runtime"] > RUNTIME_MAX)
).sum()
df_clean.loc[
    (df_clean["runtime"] < RUNTIME_MIN) | (df_clean["runtime"] > RUNTIME_MAX),
    "runtime",
] = np.nan
print(f"Set {n_bad_runtime} implausible runtime values → NaN "
      f"(outside [{RUNTIME_MIN}, {RUNTIME_MAX}] min)")

runtime_median = df_clean["runtime"].median()
df_clean["runtime"].fillna(runtime_median, inplace=True)
print(f"Imputed remaining NaN runtimes with median = {runtime_median:.1f} min")

# Budget / Revenue (flag 0 values)
n_zero_budget  = (df_clean["budget"]  == 0).sum()
n_zero_revenue = (df_clean["revenue"] == 0).sum()
print(f"budget  = 0 : {n_zero_budget:,}  ({100*n_zero_budget/len(df_clean):.1f}%)")
print(f"revenue = 0 : {n_zero_revenue:,} ({100*n_zero_revenue/len(df_clean):.1f}%)")
df_clean["budget_known"]  = df_clean["budget"]  > 0
df_clean["revenue_known"] = df_clean["revenue"] > 0

# Popularity Outliers (3 * IQR Range)
df_clean["popularity"] = pd.to_numeric(df_clean["popularity"], errors="coerce")
Q1, Q3 = df_clean["popularity"].quantile([0.25, 0.75])
upper = Q3 + 3 * (Q3 - Q1)
n_pop_outlier = (df_clean["popularity"] > upper).sum()
df_clean["popularity"] = df_clean["popularity"].clip(upper=upper)
print(f"Capped {n_pop_outlier} outliers above {upper:.3f}")

# Status Imputation
mode_status = df_clean["status"].mode()[0]
n_miss = df_clean["status"].isna().sum()
df_clean["status"].fillna(mode_status, inplace=True)
print(f"\nStatus: filled {n_miss} NaN → '{mode_status}' (mode)")

# Original Language Imputation
df_clean["original_language"].fillna("en", inplace=True)
print("original_language: filled NaN → 'en'")

# Dropped Rows with No Title
n_before = len(df_clean)
df_clean.dropna(subset=["title"], inplace=True)
print(f"Rows dropped (missing title): {n_before - len(df_clean)}")

print(f"\nShape after cleaning : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} cols")

# Feature Engineering
# Helper Functions
def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ("[]", "{}"):
        return []
    try:
        return ast.literal_eval(str(val))
    except Exception:
        try:
            return json.loads(str(val).replace("'", '"'))
        except Exception:
            return []

def extract_names(val):
    parsed = safe_parse(val)
    if isinstance(parsed, dict):
        return [parsed.get("name", "")]
    return [d.get("name", "") for d in parsed if isinstance(d, dict)]

# Parse JSON-esque Columns
df_clean["genres_list"] = df_clean["genres"].apply(extract_names)
df_clean["companies_list"] = df_clean["production_companies"].apply(extract_names)
df_clean["countries_list"] = df_clean["production_countries"].apply(extract_names)

df_clean["n_genres"] = df_clean["genres_list"].apply(len)
df_clean["n_companies"] = df_clean["companies_list"].apply(len)
df_clean["n_countries"] = df_clean["countries_list"].apply(len)

df_clean["primary_genre"] = df_clean["genres_list"].apply(
    lambda x: x[0] if x else "Unknown"
)
df_clean["in_collection"] = df_clean["belongs_to_collection"].apply(
    lambda x: not pd.isna(x) and str(x).strip() not in ("", "[]", "{}")
)
print("Parsed genres, production_companies, production_countries, collection")

# Date Features
df_clean["release_year"] = df_clean["release_date"].dt.year.astype("Int64")
df_clean["release_month"] = df_clean["release_date"].dt.month.astype("Int64")
df_clean["release_quarter"] = df_clean["release_date"].dt.quarter.astype("Int64")

def month_to_season(m):
    if pd.isna(m): return "Unknown"
    m = int(m)
    if m in (12, 1, 2): return "Winter"
    if m in (3, 4, 5): return "Spring"
    if m in (6, 7, 8): return "Summer"
    return "Fall"

df_clean["release_season"] = df_clean["release_month"].apply(month_to_season)
df_clean["is_summer_release"] = df_clean["release_month"].isin([6, 7, 8])
df_clean["is_holiday_release"]= df_clean["release_month"].isin([11, 12])
df_clean["movie_age"] = 2024 - df_clean["release_year"].astype(float)
df_clean.loc[df_clean["movie_age"] < 0, "movie_age"] = np.nan
print("Date features: year, month, quarter, season, movie_age, summer/holiday flags")

# Financial Features
df_clean["log_budget"] = np.where(df_clean["budget"]  > 0,
                                   np.log1p(df_clean["budget"]),  0)
df_clean["log_revenue"] = np.where(df_clean["revenue"] > 0,
                                   np.log1p(df_clean["revenue"]), 0)

mask = df_clean["budget_known"] & df_clean["revenue_known"]
df_clean["profit"] = np.where(mask,
    df_clean["revenue"] - df_clean["budget"], np.nan)
df_clean["roi"] = np.where(
    mask & (df_clean["budget"] > 0),
    (df_clean["revenue"] - df_clean["budget"]) / df_clean["budget"],
    np.nan,
)
df_clean["is_profitable"] = np.where(
    df_clean["profit"].notna(), df_clean["profit"] > 0, np.nan
)
print("Financial: log_budget, log_revenue, profit, roi, is_profitable")

# Text-based Features
df_clean["title_word_count"] = df_clean["title"].str.split().apply(len)
df_clean["title_char_count"] = df_clean["title"].str.len()
df_clean["has_tagline"] = df_clean["tagline"].notna()
df_clean["has_homepage"] = df_clean["homepage"].notna()
df_clean["has_overview"] = df_clean["overview"].notna()
df_clean["overview_word_count"] = (
    df_clean["overview"].fillna("").str.split().apply(len)
)
print("Text: title_word_count, title_char_count, has_tagline/homepage/overview, overview_word_count")

# Language Features
df_clean["is_english"] = df_clean["original_language"] == "en"
df_clean["is_multilingual"] = df_clean["n_countries"] > 1
print("Language: is_english, is_multilingual")

# Runtime Bins
df_clean["runtime_bin"] = pd.cut(
    df_clean["runtime"],
    bins=[0, 60, 90, 120, 150, 600],
    labels=["<60", "60-90", "90-120", "120-150", ">150"],
)
print("Runtime: runtime_bin (<60 / 60-90 / 90-120 / 120-150 / >150 min)")

# Popularity (sorted into Tiers based on Quartile)
df_clean["popularity_tier"] = pd.qcut(
    df_clean["popularity"], q=4,
    labels=["Low", "Medium", "High", "Blockbuster"],
    duplicates="drop",
)
print("Popularity: popularity_tier (quartile-based)")

# Major Studio Flag (has it been produced by a major studio or not)
company_freq  = df_clean["companies_list"].explode().dropna().value_counts()
MAJOR_STUDIOS = set(company_freq[company_freq >= 10].index)
df_clean["has_major_studio"] = df_clean["companies_list"].apply(
    lambda lst: any(c in MAJOR_STUDIOS for c in lst)
)
print("Studio: has_major_studio (company appeared in ≥10 films)")

# Genre (One-Hot Encoding)
ALL_GENRES = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "Thriller", "War", "Western",
]
for genre in ALL_GENRES:
    col_name = f"genre_{genre.replace(' ', '_').lower()}"
    df_clean[col_name] = df_clean["genres_list"].apply(lambda g, genre=genre: genre in g)
print(f"Genre dummies: {len(ALL_GENRES)} binary columns (genre_*)")

# Language (One-Hot Encoding)
top_langs = df_clean["original_language"].value_counts().head(6).index.tolist()
for lang in top_langs:
    df_clean[f"lang_{lang}"] = df_clean["original_language"] == lang
print(f"Language dummies: {len(top_langs)} columns (lang_*)")

# Budget/Revenue Tiers
df_clean["budget_tier"] = pd.cut(
    df_clean["budget"],
    bins=[-1, 0, 5e6, 20e6, 60e6, 1e12],
    labels=["Unknown", "Low", "Mid", "High", "Blockbuster"],
)
print("Budget tier: Unknown / Low / Mid / High / Blockbuster")

# Post-cleaning Validation
# Check to see if target attributes were changed
orig_va = df["vote_average"].reindex(df_clean.index)
orig_vc = df["vote_count"].reindex(df_clean.index)
assert (df_clean["vote_average"].isna() == orig_va.isna()).all(), "vote_average NaN pattern changed!"
assert (df_clean["vote_count"].isna()   == orig_vc.isna()).all(), "vote_count NaN pattern changed!"

# Check non-NaN values are numerically equal
va_mask = orig_va.notna()
assert np.allclose(
    df_clean.loc[va_mask, "vote_average"].astype(float),
    orig_va[va_mask].astype(float)
), "vote_average values changed!"
print("\nvote_average is unchanged")
vc_mask = orig_vc.notna()
assert np.allclose(
    df_clean.loc[vc_mask, "vote_count"].astype(float),
    orig_vc[vc_mask].astype(float)
), "vote_count values changed!"
print("vote_count   is unchanged")

remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"\nRemaining columns with NaN ({len(remaining_missing)}):")
print(remaining_missing.to_string())
print(f"\nFinal shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

# Correlation Analysis
feat_cols = [
    "budget", "revenue", "runtime", "popularity",
    "log_budget", "log_revenue", "profit", "roi",
    "n_genres", "n_companies", "n_countries",
    "title_word_count", "overview_word_count",
    "movie_age", "is_english", "in_collection",
    "has_major_studio", "has_tagline", "has_homepage",
    "is_summer_release", "is_holiday_release",
    "vote_average", "vote_count",
]
feat_cols = [c for c in feat_cols if c in df_clean.columns]

corr_df = (
    df_clean[feat_cols]
    .apply(pd.to_numeric, errors="coerce")
    .corr()
)
target_corr = (
    corr_df[["vote_average", "vote_count"]]
    .drop(["vote_average", "vote_count"])
    .sort_values("vote_average")
)
print("\nTop correlations with vote_average (engineered features):")
print(target_corr.to_string())

# Figure 2
fig, axes = plt.subplots(1, 2, figsize=(24, 18))
fig.suptitle("Feature Correlations", fontsize=14, fontweight="bold")

sns.heatmap(
    corr_df, ax=axes[0], cmap="coolwarm", center=0,
    fmt=".2f", annot=True, annot_kws={"size": 7},
    linewidths=0.3, square=True,
)
axes[0].set_title("Full Correlation Matrix")
axes[0].tick_params(axis="x", rotation=45, labelsize=8)
axes[0].tick_params(axis="y", labelsize=8)

colors = ["#C44E52" if v < 0 else "#4C72B0"
          for v in target_corr["vote_average"]]
axes[1].barh(target_corr.index, target_corr["vote_average"], color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Correlation with vote_average\n(engineered features)")
axes[1].set_xlabel("Pearson r")

plt.tight_layout()
plt.savefig("feature_correlations.png", dpi=130, bbox_inches="tight")
plt.close()
print("Saved figure as feature_correlations.png");

# Save Output
drop_list_cols = ["genres_list", "companies_list", "countries_list"]
df_out = df_clean.drop(columns=drop_list_cols, errors="ignore")
df_out.to_csv("movies_clean.csv", index=False)
print(f"\nSaved: movies_clean.csv  ({df_out.shape[0]:,} rows × {df_out.shape[1]} cols)")

new_features = [c for c in df_out.columns if c not in df.columns]
print(f"\n{len(new_features)} new features engineered:\n")
for f in sorted(new_features):
    print(f"{f:<40}  ({str(df_out[f].dtype)})")

# Study 1

In [ ]:
# Study 1

# Load the clean dataset
df = pd.read_csv("movies_clean.csv", low_memory=False)
print(f"Loaded {len(df):,} movies | {df.shape[1]} columns")
 
# Parse genres once (for convenience)
GENRE_COLS = [c for c in df.columns if c.startswith("genre_")]
 
def genre_set(row):
    """Return frozenset of genre names for a row using pre-encoded dummies."""
    return frozenset(
        c.replace("genre_", "").replace("_", " ").title()
        for c in GENRE_COLS if row[c]
    )
 
df["_genre_set"] = df.apply(genre_set, axis=1)

# Jaccard Similarity
def jaccard(set_a: frozenset, set_b: frozenset) -> float:
    """
    |A ∩ B| / |A ∪ B|
    Returns 0 if both sets are empty.
    """
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)
 
 
# Euclidean Similarity
def euclidean_similarity(value_a: float, value_b: float,
                          scale: float) -> float:
    """
    sim = 1 / (1 + ||a - b|| / scale)
    scale is the range (max - min) of the attribute, used for normalisation.
    """
    dist = abs(value_a - value_b) / scale
    return 1.0 / (1.0 + dist)
 
 
# Manhattan Similarity
def manhattan_similarity(value_a: float, value_b: float,
                          scale: float) -> float:
    """
    For 1-D inputs Euclidean == Manhattan, so we apply a different conversion:
    sim = 1 - |a - b| / scale   (clamped to [0,1])
    This gives a linear decay rather than hyperbolic, making it meaningfully
    different from the Euclidean formula above.
    """
    dist = abs(value_a - value_b) / scale
    return max(0.0, 1.0 - dist)
 
 
# Edit-Distance Similarity
def levenshtein(s: str, t: str) -> int:
    """Classic DP Levenshtein edit distance (insert, delete, substitute)."""
    s, t = s.lower(), t.lower()
    m, n = len(s), len(t)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            if s[i - 1] == t[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = temp
    return dp[n]
 
 
def edit_distance_similarity(s: str, t: str) -> float:
    """
    Normalised edit-distance similarity:
    sim = 1 - edit_distance(s, t) / max(len(s), len(t))
    Returns 1.0 for identical strings, 0.0 for completely different ones.
    """
    if not s and not t:
        return 1.0
    denom = max(len(s.lower()), len(t.lower()))
    if denom == 0:
        return 1.0
    return 1.0 - levenshtein(s, t) / denom
 
 
# Cosine Similarity
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """
    cos(θ) = (a · b) / (||a|| × ||b||)
    Returns 0 if either vector is the zero vector.
    """
    dot   = float(np.dot(vec_a, vec_b))
    norm  = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return dot / norm if norm > 0 else 0.0

DISPLAY_COLS = ["title", "primary_genre", "release_year",
                "runtime", "vote_average", "popularity",
                "budget", "revenue"]

# Helper function which builds a "Top 10" rsult table
def top10_table(scores: pd.Series, query_idx: int,
                label: str) -> pd.DataFrame:
    """
    Given a Series of similarity scores (index = df index),
    return the top-10 excluding the query itself,
    with tie-breaking on vote_average then popularity.
    """
    result = (
        df.assign(_sim=scores)
        .drop(index=query_idx, errors="ignore")
        .sort_values(["_sim", "vote_average", "popularity"],
                     ascending=[False, False, False])
        .head(10)
        [DISPLAY_COLS + ["_sim"]]
        .reset_index(drop=True)
    )
    result.index += 1 # rank from 1
    result.rename(columns={"_sim": label}, inplace=True)
    return result

# Scales for Numerical Measures
rev_scale = df["revenue"].max() - df["revenue"].min()
runtime_scale = df["runtime"].max() - df["runtime"].min()
 
# Budget: [log_budget, log_revenue] matrix for cosine
bmat = df[["log_budget", "log_revenue"]].fillna(0).to_numpy()

# Jaccard Query - Genre
QUERY1_IDX = df[df["title"] == "Toy Story"].index[0]
q1 = df.loc[QUERY1_IDX]
q1_genres = q1["_genre_set"]
 
jaccard_scores = df["_genre_set"].apply(lambda g: jaccard(q1_genres, g))

print("GENRE SIMILARITY (Jaccard) TO TOY STORY")
t1 = top10_table(jaccard_scores, QUERY1_IDX, "jaccard_sim")
print(t1[["title","primary_genre","release_year","runtime",
          "vote_average","popularity","jaccard_sim"]].to_string())
 
# Euclidean Query - Genre
QUERY2_IDX = df[df["title"] == "Titanic"].index[0]
q2 = df.loc[QUERY2_IDX]
q2_rev = float(q2["revenue"])
 
eucl_scores = df["revenue"].apply(
    lambda r: euclidean_similarity(q2_rev, float(r), rev_scale)
)
 
print("REVENUE SIMILARITY (Euclidean) TO TITANIC")
t2 = top10_table(eucl_scores, QUERY2_IDX, "euclidean_sim")
print(t2[["title","primary_genre","release_year","revenue",
          "vote_average","popularity","euclidean_sim"]].to_string())
 
# Manhattan Query - Runtime
QUERY3_IDX = df[df["title"] == "Mortal Kombat"].index[0]
q3 = df.loc[QUERY3_IDX]
q3_rt = float(q3["runtime"])
 
manh_scores = df["runtime"].apply(
    lambda r: manhattan_similarity(q3_rt, float(r), runtime_scale)
)
 
print("RUNTIME SIMILARITY (Manhattan) TO MORTAL KOMBAT")
t3 = top10_table(manh_scores, QUERY3_IDX, "manhattan_sim")
print(t3[["title","primary_genre","release_year","runtime",
           "manhattan_sim"]].to_string())
 
# Edit Distance Query - Title
QUERY4_IDX = df[df["title"] == "Se7en"].index[0]
q4 = df.loc[QUERY4_IDX]
q4_title = str(q4["title"])
 
edit_scores = df["title"].astype(str).apply(
    lambda t: edit_distance_similarity(q4_title, t)
)

print("TITLE SIMILARITY (Levenshtein Edit Distance) TO SE7EN")
t4 = top10_table(edit_scores, QUERY4_IDX, "edit_dist_sim")
t4["edit_dist"] = t4["title"].apply(
    lambda t: levenshtein(q4_title, str(t))
)
print(t4[["title","primary_genre","release_year","vote_average",
          "popularity","edit_dist","edit_dist_sim"]].to_string())
 
# Cosine Query - Budget
QUERY5_IDX = df[df["title"] == "Friday"].index[0]
q5 = df.loc[QUERY5_IDX]
q5_vec = bmat[QUERY5_IDX]
 
cosine_scores = pd.Series(
    [cosine_similarity(q5_vec, bmat[i]) for i in range(len(df))],
    index=df.index
)

print("BUDGET SIMILARITY (Cosine on [log_budget, log_revenue]) TO FRIDAY")
t5 = top10_table(cosine_scores, QUERY5_IDX, "cosine_sim")
print(t5[["title","primary_genre","release_year","budget","revenue",
          "vote_average","cosine_sim"]].to_string())

<h3><b>Analysis of Study 1</b></h3>
<p>
This study demonstrates how different similarity measures capture distinct notions of “similarity” depending on the attribute type.

For <b>categorical</b> data, the <b>Jaccard</b> similarity on genre produced rather intuitive results, returning exclusively Animation/Family films for Toy Story, all with perfect similarity (1.0). This shows that set-based measures work well when attributes are discrete and clearly defined, though they may lack granularity when categories fully match.

For <b>numerical</b> attributes, both <b>Euclidean</b> (revenue) and <b>Manhattan</b> (runtime) similarities highlight a limitation: results are driven purely by magnitude. In the revenue query, high-grossing blockbuster films dominate, even though they are not semantically related to Titanic. Similarly, runtime similarity returns films with identical duration but vastly different genres and themes. This indicates that numerical similarity is not reliable when reflecting meaningful content similarity.

<b>Text-based</b> similarity using <b>Levenshtein distance</b> provides more nuanced results. Titles like Seven and Selena are identified as similar to Se7en due to small character differences. However, this also reveals a weakness: lexical similarity does not imply thematic similarity, as many returned films are unrelated in genre or content.

Finally, <b>cosine</b> similarity on budget and revenue (after log transformation) captures proportional financial similarity rather than absolute values. The results include films with similar production scale and financial structure, even across different genres and time periods (assuming proper scaling wrt inflation). This demonstrates the strength of vector-based similarity in identifying patterns across multiple numerical features. However, much like other similarity measures, this similarity measure does not serve as a proper method of idenitfying similar films.

Overall, the experiment highlights that no <b>single similarity measure is universally optimal</b>. Categorical measures are precise but rigid, numerical measures are scalable but context-agnostic, and text-based measures capture surface-level similarity. Combining multiple attributes and similarity measures would likely yield more meaningful recommendations. Independently, they can be useful and are already present in everyday software (searching by title, recommended list based off of what you watch, etc.).
</p>

# Study 2

In [ ]:
# Study 2

# Aesthetic Color Palette
PALETTE = [
    "#7F77DD","#1D9E75","#D85A30","#378ADD","#EF9F27",
    "#E24B4A","#639922","#D4537E","#888780","#534AB7"
]
NOISE_COLOR = "#cccccc"

# Load the cleaned dataset
df = pd.read_csv("movies_clean.csv", low_memory=False)
print(f"Loaded {len(df):,} rows")

# First Combination: budget_known and revenue_known
mask_A = df["budget_known"] & df["revenue_known"]
dfA = df[mask_A][
    ["title","primary_genre","log_budget","log_revenue",
     "budget","revenue","popularity","vote_average","runtime","release_year"]
].dropna().reset_index(drop=True)
 
# Second Combination: budget_known and runtime
mask_B = df["budget_known"] & df["runtime"].notna()
dfB = df[mask_B][
    ["title","primary_genre","log_budget","runtime",
     "budget","revenue","popularity","vote_average","release_year"]
].dropna().reset_index(drop=True)
 
print(f"Combination A (log_budget vs log_revenue) : {len(dfA):,} points")
print(f"Combination B (log_budget vs runtime)     : {len(dfB):,} points")
 
# Scaling
scaler_A = StandardScaler()
XA = scaler_A.fit_transform(dfA[["log_budget","log_revenue"]])
 
scaler_B = StandardScaler()
XB = scaler_B.fit_transform(dfB[["log_budget","runtime"]])

# KMeans Clustering
# Range of values for k
K_VALUES = [2, 3, 4, 5, 6, 8]
 
def run_kmeans_all_k(X, label):
    results = {}
    inertias = []
    for k in K_VALUES:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertia = km.inertia_
        inertias.append(inertia)
        sil = silhouette_score(X, labels) if k > 1 else float("nan")
        db  = davies_bouldin_score(X, labels) if k > 1 else float("nan")
 
        # Cluster Cardinality and Magnitude
        card = {int(c): int((labels==c).sum()) for c in np.unique(labels)}
        mag  = {}
        for c in np.unique(labels):
            pts = X[labels==c]
            if len(pts) > 1:
                centroid = pts.mean(axis=0, keepdims=True)
                mag[int(c)] = float(cdist(pts, centroid).mean())
            else:
                mag[int(c)] = 0.0
 
        results[k] = {
            "labels": labels.tolist(),
            "centroids": km.cluster_centers_.tolist(),
            "inertia": float(inertia),
            "silhouette": float(sil) if not np.isnan(sil) else None,
            "davies_bouldin": float(db) if not np.isnan(db) else None,
            "cardinality": card,
            "magnitude": mag,
        }
        print(f"  {label} k={k:2d} | inertia={inertia:9.1f} | sil={sil:.4f} | DB={db:.4f} | sizes={list(card.values())}")
 
    return results, inertias
 
km_A, inertias_A = run_kmeans_all_k(XA, "Combination A")
km_B, inertias_B = run_kmeans_all_k(XB, "Combination B")
 
# Add the raw coordinates back for export
for k, res in km_A.items():
    res["x"] = dfA["log_budget"].tolist()
    res["y"] = dfA["log_revenue"].tolist()
    res["title"] = dfA["title"].tolist()
    res["genre"] = dfA["primary_genre"].tolist()
    res["budget"] = dfA["budget"].tolist()
    res["revenue"] = dfA["revenue"].tolist()
    res["vote_average"] = dfA["vote_average"].tolist()
 
for k, res in km_B.items():
    res["x"] = dfB["log_budget"].tolist()
    res["y"] = dfB["runtime"].tolist()
    res["title"] = dfB["title"].tolist()
    res["genre"] = dfB["primary_genre"].tolist()
    res["budget"] = dfB["budget"].tolist()
    res["revenue"] = dfB["revenue"].tolist()
    res["vote_average"] = dfB["vote_average"].tolist()

# DBScan
# Varying DBScan parameters
DBSCAN_GRID = [
    {"eps": 0.3, "min_samples": 5},
    {"eps": 0.3, "min_samples": 15},
    {"eps": 0.5, "min_samples": 5},
    {"eps": 0.5, "min_samples": 15},
    {"eps": 0.8, "min_samples": 5},
    {"eps": 0.8, "min_samples": 15},
    {"eps": 1.2, "min_samples": 5},
    {"eps": 1.2, "min_samples": 15},
]

def run_dbscan_grid(X, label):
    results = {}
    for cfg in DBSCAN_GRID:
        eps, ms = cfg["eps"], cfg["min_samples"]
        db  = DBSCAN(eps=eps, min_samples=ms)
        lbl = db.fit_predict(X)
 
        unique   = np.unique(lbl)
        n_clusters = len(unique[unique >= 0])
        n_noise    = int((lbl == -1).sum())
        noise_frac = n_noise / len(lbl)
 
        sil = db_score = float("nan")
        if n_clusters > 1 and n_noise < len(lbl):
            mask = lbl != -1
            if mask.sum() > n_clusters:
                sil      = silhouette_score(X[mask], lbl[mask])
                db_score = davies_bouldin_score(X[mask], lbl[mask])
 
        card = {int(c): int((lbl==c).sum()) for c in unique}
        key  = f"eps{eps}_ms{ms}"
        results[key] = {
            "eps": eps, "min_samples": ms,
            "labels": lbl.tolist(),
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "noise_frac": float(noise_frac),
            "silhouette": float(sil) if not np.isnan(sil) else None,
            "davies_bouldin": float(db_score) if not np.isnan(db_score) else None,
            "cardinality": {str(k): v for k, v in card.items()},
        }
        noise_pct = f"{100*noise_frac:.1f}%"
        sil_str = f"{sil:.4f}" if not np.isnan(sil) else " n/a "
        print(f"{label} eps={eps} ms={ms:2d} | clusters={n_clusters:2d} | noise={n_noise:4d}({noise_pct}) | sil={sil_str}")
 
    return results
 
dbs_A = run_dbscan_grid(XA, "Combination A")
dbs_B = run_dbscan_grid(XB, "Combination B")
 
for key, res in dbs_A.items():
    res["x"] = dfA["log_budget"].tolist()
    res["y"] = dfA["log_revenue"].tolist()
    res["title"] = dfA["title"].tolist()
    res["genre"] = dfA["primary_genre"].tolist()
    res["budget"] = dfA["budget"].tolist()
    res["revenue"] = dfA["revenue"].tolist()
    res["vote_average"] = dfA["vote_average"].tolist()
 
for key, res in dbs_B.items():
    res["x"] = dfB["log_budget"].tolist()
    res["y"] = dfB["runtime"].tolist()
    res["title"] = dfB["title"].tolist()
    res["genre"] = dfB["primary_genre"].tolist()
    res["budget"] = dfB["budget"].tolist()
    res["revenue"] = dfB["revenue"].tolist()
    res["vote_average"] = dfB["vote_average"].tolist()

# Figures
def scatter_cluster(ax, X_raw, labels, title,
                    xlabel, ylabel, highlight_centroids=None,
                    alpha=0.45, s=14):
    unique = sorted(set(labels))
    for i, c in enumerate(unique):
        mask = labels == c
        col  = NOISE_COLOR if c == -1 else PALETTE[i % len(PALETTE)]
        lbl  = "Noise" if c == -1 else f"Cluster {c}"
        ax.scatter(X_raw[mask, 0], X_raw[mask, 1],
                   c=col, s=s, alpha=alpha, linewidths=0,
                   label=lbl, rasterized=True)
    if highlight_centroids is not None:
        for i, cen in enumerate(highlight_centroids):
            ax.scatter(*cen, marker="*", s=180,
                       c=PALETTE[i % len(PALETTE)],
                       edgecolors="white", linewidths=0.8, zorder=5)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, pad=6)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, markerscale=1.5,
              loc="upper left", framealpha=0.8)
 
# Figure 1: KMeans Elbow Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("KMeans – Elbow Curves (Inertia vs k)", fontsize=12, fontweight="bold")
 
for ax, inertias, title in [
    (axes[0], inertias_A, "Combination A: log_budget vs log_revenue"),
    (axes[1], inertias_B, "Combination B: log_budget vs runtime"),
]:
    ax.plot(K_VALUES, inertias, "o-", color="#7F77DD", linewidth=2, markersize=7)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("k")
    ax.set_ylabel("Inertia (within-cluster SSE)")
    ax.set_xticks(K_VALUES)
    ax.tick_params(labelsize=9)
    # annotate inertia values
    for k, iner in zip(K_VALUES, inertias):
        ax.annotate(f"{iner:.0f}", (k, iner),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=7.5, color="grey")
 
plt.tight_layout()
plt.savefig("kmeans_elbow.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: kmeans_elbow.png")
 
# Figure 2: KMeans Cluster Scatter (for both Combinations) (k = 3, 5)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("KMeans Cluster Visualisation", fontsize=13, fontweight="bold")
 
SHOW_K = [3, 5]
for col_i, k in enumerate(SHOW_K):
    # Row 1: Combination A
    res   = km_A[k]
    lbl   = np.array(res["labels"])
    cens_A = scaler_A.inverse_transform(res["centroids"])
    scatter_cluster(axes[0, col_i], dfA[["log_budget","log_revenue"]].to_numpy(),
                    lbl, f"Combination A – k={k}\nlog_budget vs log_revenue",
                    "log(budget)", "log(revenue)",
                    highlight_centroids=cens_A)
    axes[0, col_i].text(0.97, 0.03,
        f"Sil={res['silhouette']:.3f}  DB={res['davies_bouldin']:.3f}",
        transform=axes[0,col_i].transAxes,
        ha="right", va="bottom", fontsize=8, color="grey")
 
    # Row 2: Combination B
    res  = km_B[k]
    lbl  = np.array(res["labels"])
    cens_B_scaled = np.array(res["centroids"])
    cens_B = scaler_B.inverse_transform(cens_B_scaled)
    scatter_cluster(axes[1, col_i], dfB[["log_budget","runtime"]].to_numpy(),
                    lbl, f"Combination B – k={k}\nlog_budget vs runtime",
                    "log(budget)", "runtime (min)",
                    highlight_centroids=cens_B)
    axes[1, col_i].text(0.97, 0.03,
        f"Sil={res['silhouette']:.3f}  DB={res['davies_bouldin']:.3f}",
        transform=axes[1,col_i].transAxes,
        ha="right", va="bottom", fontsize=8, color="grey")
 
plt.tight_layout()
plt.savefig("kmeans_clusters.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: kmeans_clusters.png")
 
# Figure 3: DBScan Scatter for both Combinations (4 configurations)
SHOW_DBS = ["eps0.3_ms5", "eps0.5_ms5", "eps0.5_ms15", "eps0.8_ms15"]
xlabels  = ["log(budget)","log(budget)","log(budget)","log(budget)"]
ylabels_A = ["log(revenue)"]*4
ylabels_B = ["runtime (min)"]*4
 
for combo_tag, dbs_results, raw_df, cols, yl_list, fname in [
    ("A", dbs_A, dfA, ["log_budget","log_revenue"], ylabels_A, "dbscan_A.png"),
    ("B", dbs_B, dfB, ["log_budget","runtime"],     ylabels_B, "dbscan_B.png"),
]:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"DBSCAN – Combination {combo_tag} Cluster Visualisation", fontsize=13, fontweight="bold")
    for ax, key in zip(axes.ravel(), SHOW_DBS):
        res  = dbs_results[key]
        lbl  = np.array(res["labels"])
        cfg  = f"eps={res['eps']} min_samples={res['min_samples']}"
        nc   = res["n_clusters"]
        nn   = res["n_noise"]
        sil  = f"{res['silhouette']:.3f}" if res['silhouette'] else "n/a"
        scatter_cluster(ax, raw_df[cols].to_numpy(), lbl,
                        f"{cfg}\nclusters={nc}, noise={nn}",
                        "log(budget)", yl_list[0], alpha=0.5)
        ax.text(0.97, 0.03, f"Sil={sil}",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=8, color="grey")
    plt.tight_layout()
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {fname}")
 
# Figure 4: Side-by-side Comparison between Best KMeans and Best DBScan
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Best Configuration: KMeans (k=4) vs DBSCAN (eps=0.5, ms=5)",
             fontsize=13, fontweight="bold")
 
BEST_K   = 4
BEST_DBS = "eps0.5_ms5"
 
for row, (km_res, dbs_res, raw_df, cols, xlabel, ylabel, tag) in enumerate([
    (km_A[BEST_K],  dbs_A[BEST_DBS], dfA, ["log_budget","log_revenue"],
     "log(budget)", "log(revenue)", "Combination A"),
    (km_B[BEST_K],  dbs_B[BEST_DBS], dfB, ["log_budget","runtime"],
     "log(budget)", "runtime (min)", "Combination B"),
]):
    lbl_km  = np.array(km_res["labels"])
    lbl_dbs = np.array(dbs_res["labels"])
 
    cens_raw = (scaler_A if row==0 else scaler_B).inverse_transform(km_res["centroids"])
 
    scatter_cluster(axes[row, 0], raw_df[cols].to_numpy(), lbl_km,
                    f"KMeans k={BEST_K} – {tag}", xlabel, ylabel,
                    highlight_centroids=cens_raw)
    axes[row, 0].text(0.97, 0.03,
        f"Sil={km_res['silhouette']:.3f}",
        transform=axes[row,0].transAxes, ha="right", va="bottom", fontsize=8, color="grey")
 
    scatter_cluster(axes[row, 1], raw_df[cols].to_numpy(), lbl_dbs,
                    f"DBSCAN eps={dbs_res['eps']} ms={dbs_res['min_samples']} – {tag}",
                    xlabel, ylabel)
    sil_str = f"{dbs_res['silhouette']:.3f}" if dbs_res['silhouette'] else "n/a"
    axes[row, 1].text(0.97, 0.03,
        f"Sil={sil_str}  Noise={dbs_res['n_noise']}",
        transform=axes[row,1].transAxes, ha="right", va="bottom", fontsize=8, color="grey")
 
plt.tight_layout()
plt.savefig("comparison.png", dpi=130, bbox_inches="tight")
plt.close()
print("Saved: comparison.png")
 
# Figure 5: Silhouette and DB score comparisons (KMeans)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("KMeans – Silhouette & Davies-Bouldin Scores vs k", fontsize=12, fontweight="bold")
 
for ax_i, (km_res, tag) in enumerate([(km_A, "Combination A"), (km_B, "Combination B")]):
    ax = axes[ax_i]
    ks   = K_VALUES[1:]   # sil undefined for k=1
    sils = [km_res[k]["silhouette"] for k in ks]
    dbs  = [km_res[k]["davies_bouldin"] for k in ks]
    ax2  = ax.twinx()
    ax.plot(ks, sils, "o-", color="#7F77DD", label="Silhouette ↑", linewidth=2)
    ax2.plot(ks, dbs, "s--", color="#D85A30", label="Davies-Bouldin ↓", linewidth=2)
    ax.set_xlabel("k"); ax.set_ylabel("Silhouette", color="#7F77DD")
    ax2.set_ylabel("Davies-Bouldin", color="#D85A30")
    ax.set_title(tag, fontsize=10)
    ax.set_xticks(ks)
    lines1, lbl1 = ax.get_legend_handles_labels()
    lines2, lbl2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, lbl1+lbl2, fontsize=8)
 
plt.tight_layout()
plt.savefig("kmeans_metrics.png", dpi=130, bbox_inches="tight")
plt.close()
print("Saved: kmeans_metrics.png")

# Interactive JSON Export Artifact 
def sample_points(res, x_col, y_col, n=600):
    idx = list(range(len(res["x"])))
    if len(idx) > n:
        rng = np.random.default_rng(42)
        idx = rng.choice(idx, n, replace=False).tolist()
    return {
        "x":      [res["x"][i]      for i in idx],
        "y":      [res["y"][i]      for i in idx],
        "labels": [res["labels"][i] for i in idx],
        "titles": [res["title"][i]  for i in idx],
        "genres": [res["genre"][i]  for i in idx],
        "vote":   [res["vote_average"][i] for i in idx],
    }
 
export = {
    "combos": {
        "A": {"xlabel": "log(budget)", "ylabel": "log(revenue)", "desc": "Financial Fingerprint"},
        "B": {"xlabel": "log(budget)", "ylabel": "runtime (min)", "desc": "Production Scale vs Length"},
    },
    "kmeans": {
        "A": {
            "elbow": {"k": K_VALUES, "inertia": inertias_A},
            "configs": {
                str(k): {
                    "k": k,
                    "inertia": km_A[k]["inertia"],
                    "silhouette": km_A[k]["silhouette"],
                    "davies_bouldin": km_A[k]["davies_bouldin"],
                    "cardinality": {str(kk): vv for kk, vv in km_A[k]["cardinality"].items()},
                    "magnitude":   {str(kk): round(vv,4) for kk, vv in km_A[k]["magnitude"].items()},
                    "points": sample_points(km_A[k], "log_budget", "log_revenue"),
                }
                for k in K_VALUES
            }
        },
        "B": {
            "elbow": {"k": K_VALUES, "inertia": inertias_B},
            "configs": {
                str(k): {
                    "k": k,
                    "inertia": km_B[k]["inertia"],
                    "silhouette": km_B[k]["silhouette"],
                    "davies_bouldin": km_B[k]["davies_bouldin"],
                    "cardinality": {str(kk): vv for kk, vv in km_B[k]["cardinality"].items()},
                    "magnitude":   {str(kk): round(vv,4) for kk, vv in km_B[k]["magnitude"].items()},
                    "points": sample_points(km_B[k], "log_budget", "runtime"),
                }
                for k in K_VALUES
            }
        },
    },
    "dbscan": {
        "A": {
            key: {
                "eps": v["eps"], "min_samples": v["min_samples"],
                "n_clusters": v["n_clusters"],
                "n_noise": v["n_noise"],
                "noise_frac": round(v["noise_frac"], 4),
                "silhouette": v["silhouette"],
                "davies_bouldin": v["davies_bouldin"],
                "cardinality": v["cardinality"],
                "points": sample_points(v, "log_budget", "log_revenue"),
            }
            for key, v in dbs_A.items()
        },
        "B": {
            key: {
                "eps": v["eps"], "min_samples": v["min_samples"],
                "n_clusters": v["n_clusters"],
                "n_noise": v["n_noise"],
                "noise_frac": round(v["noise_frac"], 4),
                "silhouette": v["silhouette"],
                "davies_bouldin": v["davies_bouldin"],
                "cardinality": v["cardinality"],
                "points": sample_points(v, "log_budget", "runtime"),
            }
            for key, v in dbs_B.items()
        },
    }
}
 
with open("clustering_results.json", "w") as f:
    json.dump(export, f)
print("Saved: clustering_results.json")

# Summary
print("SUMMARY – KMeans (best k by silhouette per combination)")
for tag, km_res in [("A", km_A), ("B", km_B)]:
    best_k = max(K_VALUES[1:], key=lambda k: km_res[k]["silhouette"] or -1)
    r = km_res[best_k]
    print(f"  Combo {tag}: best k={best_k}  sil={r['silhouette']:.4f}  DB={r['davies_bouldin']:.4f}  inertia={r['inertia']:.1f}")
 
print("\nSUMMARY – DBSCAN (sorted by silhouette, Combination A & B)")
for tag, dbs_res in [("A", dbs_A), ("B", dbs_B)]:
    rows = [(k, v) for k, v in dbs_res.items() if v["silhouette"]]
    rows.sort(key=lambda t: t[1]["silhouette"], reverse=True)
    print(f"  Combination {tag}:")
    for key, v in rows[:3]:
        print(f"    eps={v['eps']} ms={v['min_samples']:2d} | "
              f"clusters={v['n_clusters']:2d} noise={v['n_noise']:4d}({100*v['noise_frac']:.1f}%) | "
              f"sil={v['silhouette']:.4f}")

<h3><b>Study 2 Analysis</b></h3>
<p>
This study evaluates two fundamental clustering algos, KMeans and DBSCAN, using two distinct attribute combinations to identify patterns in movie performance and characteristics:
    - Combination A: log_budget vs. log_revenue
    - Combination B: log_budget vs. runtime
</p>
<h4><b>KMeans Clustering Results</b></h4>
<p>
To determine the optimal number of clusters, the Elbow Method and evaluation metrics (Silhouette Score and Davies-Bouldin Index) were applied. The value of k ranged from 2 to 8. When it comes to Combination A, the elbow curve suggests k = 3 due to its peaking Silhouette score. For Combo B, the metrics are less definitive, but k = 5 provides a lower Davies-Bouldin score, suggesting better-defined clusters despite the higher density of the data. Visual Inspection: KMeans creates relatively balanced, spherical regions. However, it forces every data point into a cluster, even extreme outliers (e.g., very low-budget movies with high revenue), which can skew the centroids.
</p>
<h4><b>DBSCAN Clustering Results</b></h4>
<p>
DBSCAN was tested to evaluate its ability to handle density-based clusters and filter out noise. When it comes to hyperparameters, we tested variations of eps (0.3, 0.5, 0.8) and min_samples (5, 15). For Combination A, the configuratio of eps = 0.5 and min_samples = 5 yielded the best esults, identifying 4 clusters with a high Silhouette score (0.737) and only 34 noise points. Combination B proved more challenging due to the high data density. Increasing eps to 0.8 with min_samples = 15 helped consolidate the main mass into a single cluster while identifying 32 noise points.
Unlike KMeans, DBSCAN effectively identifies the "core" dense region of the data. it trats sparse outliers as noise, preventing them from muddying the cluster definitions.
</p>
<h4><b>Conclusion</b></h4>
<p>
DBSCAN appears more appropriate for this dataset. The movie data contains a significant number of outliers (aka "noise"). DBSCAN’s ability to ignore these points results in much higher Silhouette scores and more meaningful "core" clusters. While KMeans provides a clean visual partition, its requirement to include every point makes it less robust against the varied success of films in the dataset (aka outliers).
</p>

## Study 3


The motivation behind developing two distinct heuristics is to capture entirely different dimensions of user preference when recommending movies. Heuristic A (The "Vibe and Era" Matcher) operates on the assumption that a viewer’s primary drivers are the broad genre and the physical commitment required to watch the film. By combining the Jaccard similarity of genres with the Euclidean distance of release year and runtime, this heuristic aims to find films that physically "feel" similar—for instance, assuming that someone who enjoyed a breezy, 80-minute animated classic from the 1990s is likely looking for another nostalgic, similarly paced family film rather than a modern three-hour epic. Conversely, Heuristic B (The "Niche and Scale" Matcher) assumes that a user is more interested in specific plot elements and the cultural footprint of the film. By measuring the Jaccard overlap of specific plot keywords alongside the Manhattan distance of popularity and user ratings, this heuristic targets the narrative substance and the "hype" level of the movie. This simulates a user who cares less about the genre or era, and more about finding highly-rated, thematic matches—ensuring that blockbuster fans are recommended other massive hits, while indie-film fans are recommended other well-regarded but obscure titles.

In [ ]:


path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ("[]", "{}"): return []
    try: return ast.literal_eval(str(val))
    except: 
        try: return json.loads(str(val).replace("'", '"'))
        except: return []

def extract_names(val):
    parsed = safe_parse(val)
    if isinstance(parsed, dict): return [parsed.get("name", "")]
    return [d.get("name", "") for d in parsed if isinstance(d, dict)]


df = pd.read_csv("movies_clean.csv", low_memory=False)
df_keywords = pd.read_csv(path + "/keywords.csv")

# Merge keywords
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df_keywords["id"] = pd.to_numeric(df_keywords["id"], errors="coerce")
df_keywords["keywords_list"] = df_keywords["keywords"].apply(extract_names)
df = pd.merge(df, df_keywords[["id", "keywords_list"]], on="id", how="left")
df["keywords_list"] = df["keywords_list"].apply(lambda x: x if isinstance(x, list) else [])


# Create _genre_set
GENRE_COLS = [c for c in df.columns if c.startswith("genre_")]
def genre_set(row):
    return frozenset(
        c.replace("genre_", "").replace("_", " ").title()
        for c in GENRE_COLS if row[c]
    )
df["_genre_set"] = df.apply(genre_set, axis=1)

# Create _keyword_set
def keyword_set(row):
    return frozenset(str(k).title() for k in row["keywords_list"])
df["_keyword_set"] = df.apply(keyword_set, axis=1)

print(f"Data loaded and prepared! Shape: {df.shape}")


In [ ]:


def keyword_set(row):
    kw = row.get("keywords_list", [])
    if isinstance(kw, list):
        return frozenset(str(k).title() for k in kw)
    elif isinstance(kw, str):
        try:
            return frozenset(str(k).title() for k in ast.literal_eval(kw))
        except:
            return frozenset()
    return frozenset()

if "keywords_list" in df.columns or "keywords" in df.columns:
    df["_keyword_set"] = df.apply(keyword_set, axis=1)
else:
    print("\nWARNING: 'keywords_list' not found. Heuristic B will only use Popularity and Rating.")
    df["_keyword_set"] = df.apply(lambda x: frozenset(), axis=1)

scales = {
    'year': df["release_year"].max() - df["release_year"].min(),
    'runtime': df["runtime"].max() - df["runtime"].min(),
    'popularity': df["popularity"].max() - df["popularity"].min(),
    'rating': df["vote_average"].max() - df["vote_average"].min()
}

# 2. Define the Heuristic Functions
def heuristic_A(target: pd.Series, candidate: pd.Series, scales: dict) -> float:
    """
    60% Jaccard on Genres | 20% Euclidean on Year | 20% Euclidean on Runtime
    """
    sim_genre = jaccard(target["_genre_set"], candidate["_genre_set"])
    
    sim_year = euclidean_similarity(
        float(target["release_year"]), float(candidate["release_year"]), scales['year']
    )
    sim_runtime = euclidean_similarity(
        float(target["runtime"]), float(candidate["runtime"]), scales['runtime']
    )
    
    return (0.60 * sim_genre) + (0.20 * sim_year) + (0.20 * sim_runtime)


def heuristic_B(target: pd.Series, candidate: pd.Series, scales: dict) -> float:
    """
    50% Jaccard on Keywords | 25% Manhattan on Popularity | 25% Manhattan on Rating
    """
    sim_kw = jaccard(target["_keyword_set"], candidate["_keyword_set"])
    
    sim_pop = manhattan_similarity(
        float(target["popularity"]), float(candidate["popularity"]), scales['popularity']
    )
    sim_rating = manhattan_similarity(
        float(target["vote_average"]), float(candidate["vote_average"]), scales['rating']
    )
    
    return (0.50 * sim_kw) + (0.25 * sim_pop) + (0.25 * sim_rating)

test_movies = ["Toy Story", "Apollo 13", "Fight Club"]

for title in test_movies:
    matches = df[df["title"] == title]
    if matches.empty:
        print(f"\nCould not find '{title}' in the dataset. Skipping...")
        continue
        
    QUERY_IDX = matches.index[0]
    target_row = df.loc[QUERY_IDX]
    
    print(f"\n\n{'='*70}")
    print(f" TARGET MOVIE: {title}")
    print(f"{'='*70}")
    
    # Calculate scores for Heuristic A
    scores_A = df.apply(lambda row: heuristic_A(target_row, row, scales), axis=1)
    t_A = top10_table(scores_A, QUERY_IDX, "heuristic_A")
    
    print("\n--- HEURISTIC A (60% Genre, 20% Release Year, 20% Runtime) ---")
    cols_A = ["title", "primary_genre", "release_year", "runtime", "heuristic_A"]
    print(t_A[cols_A].to_string())
    
    
    # Calculate scores for Heuristic B
    scores_B = df.apply(lambda row: heuristic_B(target_row, row, scales), axis=1)
    t_B = top10_table(scores_B, QUERY_IDX, "heuristic_B")
    
    print("\n--- HEURISTIC B (50% Keywords, 25% Popularity, 25% Rating) ---")
    cols_B = ["title", "popularity", "vote_average", "heuristic_B"]
    print(t_B[cols_B].to_string())



### 1. Toy Story
* **Heuristic A:** Acted like a strict 1990s TV Guide. It successfully found other animated family films released around 1995 with an exact runtime of ~75 minutes (e.g., *A Flintstones Christmas Carol*, *Chicken Run*). It even successfully found *Toy Story 2*.
* **Heuristic B:** Successfully pulled in the sequels (*Toy Story 2* & *3*) and thematic matches like *The Iron Giant* and *Pinocchio*. However, it also recommended *Django Unchained* and *Ted*. This likely happened because both *Ted* and *Toy Story* share keywords like "toy" or "friendship", and because they are both massive hits, their popularity/rating scores created a mathematical match despite being wildly different genres.

### 2. Apollo 13
* **Heuristic A (The Failure):** This is where Heuristic A falls apart. Because *Apollo 13*'s primary genre is simply "Drama," Heuristic A just fetched a list of random, completely unrelated Dramas that happened to be released between 1993-1996 and were exactly 140 minutes long (*Midaq Alley*, *Secrets & Lies*). It completely missed the "space" concept.
* **Heuristic B (The Success):** This result is incredible. By relying on plot keywords rather than broad genres, it perfectly captured the highly specific "Space Survival / Competence Porn" subgenre. It recommended *Interstellar*, *The Martian*, *Gravity*, and *Sully* (another true story about a pilot crisis). 

### 3. Fight Club
* **Heuristic A:** Repeated its Apollo 13 mistake. It looked for "Dramas" from 1999 that were ~139 minutes long and suggested *Angela's Ashes* and *The Hurricane*. These have absolutely nothing in common with the gritty, dark tone of *Fight Club*. 
* **Heuristic B:** Nailed the psychological, cult-classic vibe perfectly. It suggested David Fincher's other dark thriller (*Se7en*), gritty sci-fi (*Terminator 2*, *Blade Runner*), and other psychological masterpieces about unstable protagonists (*A Clockwork Orange*, *One Flew Over the Cuckoo's Nest*). 


## Study 4

In [ ]:

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

df_metadata = pd.read_csv(path + "/movies_metadata.csv", low_memory=False)
df_ratings = pd.read_csv(path + "/ratings_small.csv")
df_links = pd.read_csv(path + "/links_small.csv")

df_metadata = df_metadata.drop_duplicates(subset=['id'])
df_metadata['id'] = pd.to_numeric(df_metadata['id'], errors='coerce')
df_metadata = df_metadata.dropna(subset=['id', 'title'])
movie_titles = df_metadata.set_index('id')['title'].to_dict()

df_links['movieId'] = pd.to_numeric(df_links['movieId'], errors='coerce')
df_links['tmdbId'] = pd.to_numeric(df_links['tmdbId'], errors='coerce')
id_to_tmdb = df_links.set_index('movieId')['tmdbId'].dropna().to_dict()

df_ratings['tmdbId'] = df_ratings['movieId'].map(id_to_tmdb)
valid_ids = set(df_metadata['id'].unique())
df_ratings = df_ratings[df_ratings['tmdbId'].isin(valid_ids)].copy()

In [ ]:

df_pivot = df_ratings.pivot(index='userId', columns='tmdbId', values='rating').fillna(0)
R_orig = df_pivot.to_numpy()
movie_ids_in_matrix = df_pivot.columns.tolist()

nonzero_indices = np.argwhere(R_orig > 0)
np.random.seed(42)
np.random.shuffle(nonzero_indices)

test_size = int(len(nonzero_indices) * 0.10)
test_indices = nonzero_indices[:test_size]

R_train = R_orig.copy()
gold_standard = []

for row, col in test_indices:
    actual = R_orig[row, col]
    gold_standard.append((row, col, actual))
    R_train[row, col] = 0 # Hide the rating from the model

print(f"Utility Matrix: {R_orig.shape[0]} users x {R_orig.shape[1]} movies")
print(f"Gold Standard: {len(gold_standard)} ratings hidden.")

In [ ]:

def matrix_factorization(R, K=20, steps=50, alpha=0.002, beta=0.02):
    N, M = R.shape
    P = np.random.rand(N, K)
    Q = np.random.rand(M, K).T
    non_zero_indices = np.argwhere(R > 0)
    
    for step in range(steps):
        for i, j in non_zero_indices:
            eij = R[i, j] - np.dot(P[i, :], Q[:, j])
            # Vectorized update for latent features
            P[i, :] += alpha * (2 * eij * Q[:, j] - beta * P[i, :])
            Q[:, j] += alpha * (2 * eij * P[i, :] - beta * Q[:, j])
            
        if step % 10 == 0:
            error = sum(pow(R[i, j] - np.dot(P[i, :], Q[:, j]), 2) for i, j in non_zero_indices)
            print(f" Step {step}/{steps} | Training Error: {error:.2f}")
    return P, Q.T


In [ ]:
# =============================================================================
# TESTING DIFFERENT LATENT DIMENSIONS (K)
# =============================================================================

k_values = [5, 10, 20, 50] # Different dimensions to test
results_log = []

for k_test in k_values:
    print(f"\n>>> Testing Latent Dimensions K = {k_test}")
    
    # 1. Train the model with the current K
    test_P, test_Q_T = matrix_factorization(R_train, K=k_test, steps=30)
    test_nR = np.dot(test_P, test_Q_T.T)
    
    # 2. Quantitative Evaluation
    # MSE
    mse_val = sum(pow(actual - test_nR[r, c], 2) for r, c, actual in gold_standard) / len(gold_standard)
    
    # Ranking Metrics
    user_results = {}
    for r, c, actual in gold_standard:
        if r not in user_results: user_results[r] = []
        user_results[r].append({'actual': actual, 'pred': test_nR[r, c]})
    
    p5_acc, mrr_acc = [], []
    for u, movies in user_results.items():
        sorted_m = sorted(movies, key=lambda x: x['pred'], reverse=True)
        rel = [1 if m['actual'] >= 4.0 else 0 for m in sorted_m]
        p5_acc.append(sum(rel[:5])/5.0)
        for rank, is_rel in enumerate(rel, 1):
            if is_rel:
                mrr_acc.append(1.0/rank)
                break
    
    # 3. Log the results
    results_log.append({
        'K': k_test,
        'MSE': mse_val,
        'P@5': np.mean(p5_acc),
        'MRR': np.mean(mrr_acc)
    })

df_results = pd.DataFrame(results_log)
print("\n" + "="*40)
print(" FINAL COMPARISON TABLE ")
print("="*40)
print(df_results.to_string(index=False))

In [ ]:

# --- Eval 1: MSE ---
mse = sum(pow(actual - nR[r, c], 2) for r, c, actual in gold_standard) / len(gold_standard)

# --- Eval 2: Ranking Metrics ---
RELEVANCE_THRESHOLD = 4.0
user_results = {}
for r, c, actual in gold_standard:
    if r not in user_results: user_results[r] = []
    user_results[r].append({'actual': actual, 'pred': nR[r, c]})

p5, p10, mrr = [], [], []
for u, movies in user_results.items():
    # Sort by predicted score
    sorted_m = sorted(movies, key=lambda x: x['pred'], reverse=True)
    rel = [1 if m['actual'] >= RELEVANCE_THRESHOLD else 0 for m in sorted_m]
    
    p5.append(sum(rel[:5])/5.0)
    p10.append(sum(rel[:10])/10.0)
    
    for rank, is_rel in enumerate(rel, 1):
        if is_rel:
            mrr.append(1.0/rank)
            break

print("\n" + "="*30 + "\n QUANTITATIVE EVALUATION \n" + "="*30)
print(f"Mean Squared Error: {mse:.4f}")
print(f"Precision@5       : {np.mean(p5):.4f}")
print(f"Precision@10      : {np.mean(p10):.4f}")
print(f"MRR               : {np.mean(mrr):.4f}")

In [ ]:

print("\n" + "="*30 + "\n SIMULATED RECOMMENDATIONS \n" + "="*30)
test_users = df_pivot.index[:3] # Taking the first 3 users in the matrix

for u_id in test_users:
    u_idx = df_pivot.index.get_loc(u_id)
    unseen = []
    for col_idx, orig_val in enumerate(R_orig[u_idx]):
        if orig_val == 0: # Only movies they haven't seen
            tmdb_id = movie_ids_in_matrix[col_idx]
            unseen.append((movie_titles.get(tmdb_id, "Unknown"), nR[u_idx, col_idx]))
    
    top_10 = sorted(unseen, key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop 10 Unseen for User {u_id}:")
    for i, (title, score) in enumerate(top_10, 1):
        print(f" {i}. {score:.2f} | {title}")

### Study 4: Collaborative Filtering Recommendation System Evaluation

**1. Introduction and Methodology**
This study evaluates a model-based collaborative filtering system implemented via **Matrix Factorization**. The system decomposes the User-Movie Utility Matrix into latent user ($P$) and movie ($Q$) features using Stochastic Gradient Descent. 



The dataset consists of **671 users** and **9,025 movies**. To rigorously test the model, a "Gold Standard" was created by randomly masking **9,981 actual ratings** (10% of the data). These ratings were hidden during training and used for evaluation. Ranking metrics were calculated using a relevance threshold of **$\ge$ 4.0 stars**.

---

**2. Latent Dimension Sensitivity Analysis**
A core objective of Study 4 was testing different values for the latent dimensions ($K$). $K$ represents the number of hidden features the model "invents" to explain user behavior. We tested $K \in \{5, 10, 20, 50\}$ to find the optimal balance between complexity and generalizability.



**Comparative Results Table:**
| Latent Dimensions (K) | MSE | P@5 | MRR |
| :--- | :--- | :--- | :--- |
| 5 | 0.8750 | 0.5570 | 0.8630 |
| **10 (Optimal)** | **0.8640** | **0.5560** | **0.8610** |
| 20 | 0.9050 | 0.5450 | 0.8520 |
| 50 | 1.0650 | 0.5350 | 0.8330 |

---

**3. Discussion and Insights**

**Predictive Accuracy and the "Sweet Spot"**
The results indicate that **$K=10$** is the optimal dimensionality for this dataset, achieving the lowest Mean Squared Error (**0.8640**). At this level, the model's predicted star ratings are, on average, within **0.93 stars** ($\sqrt{0.864}$) of the actual user ratings. This suggests that 10 hidden features are sufficient to capture the "DNA" of the movie dataset.

**The Impact of Overfitting**
A significant trend is observed as $K$ increases beyond 10. When $K$ reached 50, the MSE degraded sharply to **1.0650**. This is a classic example of **overfitting**, where the model becomes so complex that it begins to "memorize" the training data rather than learning general patterns. This results in poorer performance on the hidden Gold Standard data.



**Ranking Performance (P@k and MRR)**
* **Precision@5:** At the optimal $K=10$, the system achieved a score of **0.5560**, meaning over half of the top 5 recommendations were "hits" (rated 4+ stars).
* **Mean Reciprocal Rank (MRR):** The MRR remained exceptionally high (above **0.83**) across all tests. An MRR of **0.8610** for the optimal model indicates that the first relevant movie for almost every user appeared in the #1 or #2 spot. This proves the system is highly reliable at surfacing top-tier content instantly.

---

**4. Qualitative Analysis (Simulated Requests)**
Using the optimal $K=10$ model, requests for unseen movies were simulated for the first three users:
* **User 1:** Received high-scoring matches like *Elizabeth* (4.92) and *Das Boot* (4.26).
* **User 2:** Showed strong alignment with acclaimed cinema like *Yojimbo* and *Bone Tomahawk*, with scores frequently exceeding 5.0.
* **User 3:** Received a diverse list featuring *Seven Samurai* and *Amélie*.

The overlap in recommendations (e.g., *The Staircase* and *Seve* appearing for multiple users) suggests the model successfully identified high-quality "anchor" titles that appeal to specific latent taste profiles.

---

**5. Conclusion**
The Collaborative Filtering model demonstrated high effectiveness in both numerical prediction and list ranking. While increasing latent dimensions initially helps, the jump to $K=50$ proved counterproductive due to overfitting. The selection of **$K=10$** provides a robust, efficient, and highly accurate engine, evidenced specifically by the **0.8610 MRR**, ensuring users find movies they love at the very top of their recommendation lists.

<h3><b>References</b></h3>
<p>N/A</p>